# Customer Churn Prediction
## Notebook 4 of 8 — Feature Engineering

This project predicts which telecom customers are likely to **churn** (cancel their service) so the business can reach them with retention offers *before* they leave. It walks through the full data-science lifecycle — exploration, cleaning, EDA, feature engineering, modelling, evaluation, and a tuned final model.

**Business problem:** Winning a new customer costs far more than keeping an existing one. This telecom loses roughly **27% of its customers**, and the leadership team wants a reliable, data-driven way to flag at-risk customers early enough to act.

**Tools & techniques:** Python · pandas · NumPy · Matplotlib · Seaborn · scikit-learn · XGBoost · SMOTE (imbalanced-learn) · joblib

> **This notebook:** we turn the cleaned data into a numeric, model-ready matrix using the reusable, unit-tested `engineer_features()` function, then save it for all modelling notebooks.

**Author:** La Yaung Linn Lett  &nbsp;·&nbsp;  **Last updated:** June 2026

---

## 1. Load the cleaned data

**What:** Read the cleaned dataset from Notebook 02 and import the feature-engineering helper.

**Why:** We start from the single cleaned source so encoding is the only transformation happening here.

In [1]:
# Standard library
import sys
import warnings
from pathlib import Path

# Third-party
import pandas as pd

# Local — reusable, unit-tested preprocessing functions (see src/data_preprocessing.py)
sys.path.append(str(Path.cwd().parent))
from src.data_preprocessing import engineer_features, split_features_target

warnings.filterwarnings("ignore", category=UserWarning)

df_clean = pd.read_csv("../data/processed/telco_churn_clean.csv")
print(f"Loaded cleaned data: {df_clean.shape}")

C:\Users\Vivobook\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loaded cleaned data: (7043, 21)


## 2. What `engineer_features()` does

All the encoding logic lives in one tested function. It performs three steps, each a deliberate modelling decision:

1. **Drops `customerID` and `TotalCharges`.** `customerID` is a unique identifier (pure noise); `TotalCharges` is highly correlated with `tenure` × `MonthlyCharges` (seen in the EDA heatmap), so dropping it removes redundancy.
2. **Label-encodes the binary columns** (`gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, and the `Churn` target) to 0/1 — the most compact encoding for two-category columns.
3. **One-hot encodes the multi-category columns** (`Contract`, `PaymentMethod`, `InternetService`, etc.) with `drop_first=True` to avoid the dummy-variable trap.

Because it's a tested function, every modelling notebook gets exactly the same feature set — no silent inconsistencies.

In [2]:
df_model = engineer_features(df_clean)
print(f"Shape after feature engineering: {df_model.shape}")
df_model.head()

Shape after feature engineering: (7043, 30)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,Churn,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,0,True,...,False,False,False,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,0,False,...,False,False,False,False,False,True,False,False,False,True
2,1,0,0,0,2,1,1,53.85,1,False,...,False,False,False,False,False,False,False,False,False,True
3,1,0,0,0,45,0,0,42.30,0,True,...,True,False,False,False,False,True,False,False,False,False
4,0,0,0,0,2,1,1,70.70,1,False,...,False,False,False,False,False,False,False,False,True,False


Encoding expands the table to **30 numeric columns** (29 features + the `Churn` target). Everything is now numeric and model-ready.

## 3. Quick sanity check

**What:** Separate features (`X`) from the target (`y`) and confirm the shapes line up.

**Why:** A fast guard that the target is cleanly split out before we save and move on to modelling.

In [3]:
X, y = split_features_target(df_model)
print(f"Features X: {X.shape}")
print(f"Target y:   {y.shape}  (churn rate: {y.mean():.1%})")

Features X: (7043, 29)
Target y:   (7043,)  (churn rate: 26.5%)


**29 feature columns**, a target vector of length 7,043, and the familiar **~27% churn rate** — exactly as expected.

## 4. Save the model-ready matrix

**What:** Write the fully encoded data to `data/processed/model_ready.csv`.

**Why:** Every modelling notebook (05-08) loads this one file, so the exact same feature set is used everywhere.

In [4]:
df_model.to_csv("../data/processed/model_ready.csv", index=False)
print("Saved -> data/processed/model_ready.csv")
print(f"Final shape: {df_model.shape}")

Saved -> data/processed/model_ready.csv
Final shape: (7043, 30)


## Section conclusion — features are model-ready

- Applied the reusable, unit-tested **`engineer_features()`** function from `src/`.
- Dropped **`customerID`** (no signal) and **`TotalCharges`** (redundant), label-encoded binaries, and one-hot encoded the rest.
- Produced a clean **7,043 × 30** numeric matrix and saved it as the single input for all modelling.

**Next:** Notebook 05 builds a baseline model on this matrix.